# Lab 7.10 &mdash; Observability: Log Every Run

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Record every stage of a run in an auditable log
- Read back the ordered trace for debugging
- Measure quality over a batch (the approval rate)

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Module 6. Advanced labs end with an **optional** cell that runs the **real** library. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Failure modes & observability](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-07-10"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
You can't run unattended what you can't see (deck slide 18). **Observability** means logging every
run &mdash; the **trigger**, each **tool call &amp; observation**, the **draft**, the **validation**
result, and the **human decision**. That log lets you **debug** a bad output, **audit** what
happened, and **measure** quality over time (approval rate, edit rate). Real stacks &mdash;
**LangSmith / Langfuse** &mdash; capture exactly this; here you build the offline version.

In [ ]:
# One run produces a sequence of stage events; a batch of runs produces a metric.
print("we log: trigger -> gather -> draft -> validate -> approve, plus the human decision")

## Your Turn
Complete the `RunLog` (record + read back the stages) and `approval_rate` over a batch.

In [ ]:
class RunLog:
    def __init__(self):
        self.events = []
    def record(self, stage, detail):
        ___   # TODO: append (stage, detail) to self.events
    def stages(self):
        return ___   # TODO: just the stage names, in order

def approval_rate(decisions):
    # the fraction of runs a human approved ("send")
    return ___   # TODO: count of "send" divided by the number of decisions

try:
    log = RunLog()
    log.record('trigger', 'email 4471')
    log.record('gather', 'lookup_order -> shipped')
    log.record('draft', 'due Friday')
    log.record('validate', 'ok')
    log.record('approve', 'send')
    print('trace stages:', log.stages())
    print('approval rate:', approval_rate(['send', 'revise', 'send', 'send']))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("record appends an event", lambda: (lambda l: (l.record("trigger", "x"), len(l.events))[1])(RunLog()) == 1)
expect_true("a full run logs five stages", lambda: (lambda l: [l.record(s, "d") for s in ("trigger","gather","draft","validate","approve")] and len(l.events))(RunLog()) == 5)
expect_true("stages() returns them in order", lambda: (lambda l: [l.record(s, "d") for s in ("trigger","gather","draft")] and l.stages())(RunLog()) == ["trigger","gather","draft"])
expect_true("approval_rate is 0.5 on [send, revise]", lambda: approval_rate(["send", "revise"]) == 0.5)
expect_true("approval_rate is 1.0 when all approve", lambda: approval_rate(["send", "send"]) == 1.0)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run this against the REAL LangChain (not graded)
See the REAL callback interface LangChain exposes (LangSmith / Langfuse capture full traces). Safe to skip &mdash; it needs `pip install langchain langchain-ollama` (then
`ollama run llama3.2:1b`) or `langchain-groq` with a `GROQ_API_KEY`. If a package, model or key is
missing the cell prints a friendly note and moves on.
**The graded steps above are offline &amp; deterministic, so the lab always verifies without a model.**

In [ ]:
try:
    from langchain_core.callbacks import BaseCallbackHandler
    class PrintHandler(BaseCallbackHandler):
        def on_tool_end(self, output, **kw):
            print("tool ->", output)
    print("Real LangChain calls handlers like PrintHandler on every model/tool event.")
    print("For full run traces: set LANGCHAIN_TRACING_V2=true + LANGCHAIN_API_KEY (LangSmith),")
    print("or run Langfuse (this course's stack) and attach its callback handler.")
except Exception as e:
    print("Install langchain-core to use real callbacks -- skipping:", type(e).__name__)
print("The RunLog above already traced every stage offline.")

---
### Key takeaway
Log every run's trigger, tools, draft, validation and decision -- that's how you debug, audit and MEASURE an automation. Once you can measure it (approval rate), you can improve it. Day 5 goes deeper.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>